# 43 — Surface-target screen, **blood**: cohort fix, artifact tests, malignant vs reactive CD4

The blood twin of 42, rebuilt. The first pass ranked **TPO, MBP, ADGRE1, PTH2R, NRROS, EVC2 and
FGFR1 above CCR4**. TPO and MBP are not surface proteins; ADGRE1 is eosinophil-restricted in
humans; NRROS is myeloid. And `frac_donors_up` came out *inverted* against plausibility —
MBP 0.962, EVC2 0.936, CCR4 0.744 — which is the signature of a technical effect, because
something that scales with arm composition repeats in every patient and real inter-patient
biology does not.

Two things were wrong with that pass, and they are not the same thing.

**1. The cohort, which was wrong and is now fixed.** 54 of the 131 donor keys are `geskin2026`
lane×HTO wells — ex vivo 2-day cultures with five drug arms and one vehicle arm per lane — so
one patient was counted six times. Five more keys are **healthy donors carried under
`disease == "SS"`**, which put their CD4 cells in the *reactive* arm of a tumour-vs-reactive
comparison. Two patients appeared under two keys. Correlated wells inflate `frac_donors_up`
directly, which is the statistic the whole complaint rests on. §0b removes them: **52
independent paired donors**, diagnosis from `clin_dx`.

**2. The contamination hypothesis, which was tested and did not survive.** Blood in SS is ~92%
malignant CD4, so malignant×monocyte doublets should outnumber reactive×monocyte doublets by
about that ratio, and every minor population should preferentially contaminate the malignant
arm. §2 puts six tests to that, against an **empirical null** made of transcripts a CD4 T cell
cannot make. The null does not fire: `LYZ`, `S100A8`, `CD14`, `VCAN`, `PPBP`, `HBB` and `MS4A1`
are flat or *down* in the malignant arm, the myeloid score is **lower** there than in the
reactive arm, and the disputed genes survive trimming, winsorising, dropping lineage-positive
cells, and depth matching. §2d adds the two artifacts that are not contamination — confinement
to one cluster, and the malignancy call being a cluster label while the comparison is between
those same clusters — and rebuilds the whole screen with the arms taken from the **TCR clone**
instead. The effects hold.

So §2 excludes nothing, and says so loudly. The disputed genes are reproducible across all ten
blood studies, absent from healthy CD4 as well as from reactive CD4, and spread evenly over the
clusters. The objection to them is a judgement about **gene identity and surface topology** —
biology — and it belongs in the surfaceome plausibility layer in §7, not in a statistical
exclusion.

| § | what it does |
|---|---|
| §0b | the cohort fix: 131 donor keys → the independent set, diagnosis from `clin_dx` |
| §1 | input, on the rebuilt QC tables |
| **§2** | **the artifact tests** — six against an empirical contamination null, plus the TCR-call control |
| §3 | malignant vs reactive CD4 on the surviving genes, with per-donor figures |
| §4–§6 | off-tumour, trafficking, escape |
| §7 | the ranked table, the control check, and the surfaceome plausibility layer |
| §8 | skin ∩ blood |

**Inputs.** `jobs/run_target_screen_qc.sh --tissue blood`, which supersedes
`run_target_screen.sh` for this compartment: true-library CP10K, robust location statistics, a
contamination-filtered recompute, depth matching, per-cell couplings and the lineage-control
genes. §2d additionally needs the same job with
`--call mal_tcr --no-cluster --out-dir data/targets/by_call_mal_tcr`.

**Moved out of this notebook.** Stratification and heterogeneity are now `44`; the CITE-seq
protein-vs-transcript analysis is now `45`.

> The object is a **10,000-gene HVG subset**, so a gene absent here is absent from the screen,
> which is not the same as unexpressed. `CD4` and `TNFRSF8` (CD30) are both gone.

In [ ]:
# ============================================================================
# §0  Setup
# ============================================================================
import warnings; warnings.filterwarnings("ignore")
import json
import sys
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy import stats


def _resolve_nb_dir() -> Path:
    start = Path.cwd()
    for base in [start, *start.parents]:
        for sub in [Path("."), Path("MF")]:
            cand = base / sub
            if cand.name == "MF" and (cand / "data").exists():
                return cand.resolve()
    raise FileNotFoundError(f"could not locate MF/data from {start}")


NB_DIR = _resolve_nb_dir()
sys.path.insert(0, str(NB_DIR / "helpers"))

import surface_targets as ST
import target_screen as TS
import target_cohort as TC
import target_qc as TQ
import target_figs as TF
import target_het as TH
import cluster_malig_helpers as CM
import malignancy as MAL

# BOTH modules, before any path is read. `malignancy` defaults to skin at import, so with only
# CM.use_tissue("blood") the arms would be built from the SKIN parquets reindexed onto blood
# cell ids -- every call undefined, every arm empty, and the first symptom is an unrelated
# KeyError five cells later. (Observed 2026-10-24.) The assert below is the guard, not the
# comment.
CM.use_tissue("blood")
MAL.use_tissue("blood")
TQ.use_tissue("blood")           # which lineages contaminate THIS compartment: 42 sets skin,
                                 # and the module keeps whatever the last notebook asked for
assert MAL.CLUSTER_PQ.name.startswith("blood_"), MAL.CLUSTER_PQ
assert "LYZ" in TQ.BAND_GENES and "KRT14" not in TQ.BAND_GENES, TQ.BAND_GENES

TISSUE = "blood"
TARGETS = NB_DIR / "data" / "targets"
PREFIX = "tgt43_"
MALIG, REACT, HC_CD4 = "CD4_malignant", "CD4_reactive", "CD4_HC"
MIN_CELLS, MIN_DONORS, DETECT_FLOOR = 20, 5, 0.10

ST.TAB_PREFIX = ST.FIG_PREFIX = PREFIX
TF.rc()
pd.set_option("display.width", 240)

print("NB_DIR =", NB_DIR)
print("tissue:", CM.TISSUE, "| mal_cluster source:", MAL.CLUSTER_PQ.name,
      "| headline:", MAL.HEADLINE)

## §0b — The cohort, before any statistic

`donor` in the target tables is not a patient. Three corrections, all read from the curated
`clin_*` columns (`docs/CLINICAL_METADATA.md`), none of them guessed here:

- **ex vivo drug arms.** `geskin2026` is six 2-day culture wells per lane — REGN668/dupilumab,
  anti-IL13, anti-IL4, anti-IL13Rα1, anti-IL13Rα2 and a vehicle control. The five drug wells are
  dropped; the vehicle well stays and is flagged `is_cultured`, because dropping it would remove
  the study entirely and because a surface result should be shown with and without cultured
  material.
- **mislabelled healthy donors.** `B4__PF46–PF50` carry `disease == "SS"` in the atlas and
  `clin_dx == "HC"` in the curation. The screen builds `CD4_HC` from the atlas column, so their
  cells were sitting in `CD4_reactive` — the benign compartment *beside a tumour* — in a
  tumour-vs-reactive test. Reassigned here.
- **one patient under two keys.** `B1__P5` is the non-malignant sort of `B1__P4`;
  `H__SS2` and `D13__SZ22` are one person sampled at two centres.

Diagnosis comes from `clin_dx` throughout. The atlas `disease` column disagrees with it for
every herrera donor, all of ren2023 and two buus donors, and is not read again in this notebook.

`burden` — the malignant share of a donor's CD4 compartment, computed from the arm cell counts
themselves — is attached here. It is the one donor covariate with complete coverage, and §2 uses
it as the mechanistic test for contamination.

In [ ]:
# ============================================================================
# §0b  Cohort: which donor keys are independent patients
# ============================================================================
don = TC.blood_donor_table()
don.to_csv(ST.tab("blood_cohort"), index=True)

_raw = pd.read_parquet(TARGETS / f"{TISSUE}_target_long.parquet")
long = TC.filter_long(_raw, don)
bur = TC.burden(long)
don = don.join(bur, how="left")

print(TC.cohort_report(don, long))
print("\nwrote", ST.tab("blood_cohort"))

print("\n" + "=" * 78)
print("dropped keys, every one with its reason:")
print(don.loc[~don["cohort_keep"], ["study", "dx", "clin_ex_vivo", "clin_arm", "person",
                                    "n_cells", "drop_reason"]]
      .sort_values(["drop_reason", "study"]).to_string())

print("\nhealthy donors moved out of the reactive arm (clin_dx == HC, atlas disease != HC):")
_moved = don[don["is_hc"] & don["cohort_keep"] & don["disease"].ne("HC")]
print(_moved[["study", "disease", "dx", "n_cells"]].to_string()
      if len(_moved) else "  none")

## §1 — Input, and the four things that make blood different

1. **The object is a 10,000-gene HVG subset**, not the full 42,347. A rare-but-specific antigen
   is exactly what HVG selection discards, so a gene missing here is missing from the screen,
   which is not the same as unexpressed. `CD4` and `TNFRSF8` (CD30) are both gone, so the
   control check runs on three of four positive controls. 41 §3b lists the rest.
2. **Blood is ~92% malignant CD4 in SS.** The reactive arm barely exists in some donors — which
   is why the paired test runs on fewer donors than in skin, and why that abundance ratio is
   also the lever behind the contamination artifact §2 goes after.
3. **The CD8 arm is a CD8+NK mixture.** `nb31` labelled on `CD8A`, which NK cells also express
   as a CD8αα homodimer, so cytotoxic NK clusters came through as CD8 — the object holds only
   606 NK cells in 667,584, which is not a credible PBMC NK fraction. `32b` corrects the 12
   clusters the malignancy review actually looked at (`blood_T_celltype_corrections_v1.csv`,
   overlaid by `CM.read_obs`) and reports ~100 more with the same signature that are **not**
   corrected. The malignant-vs-reactive CD4 axis is unaffected; any statement about the CD8 or
   NK arm in §4 is a statement about a mixture. The fix belongs in `nb31`.
4. **The tables are the QC build.** `mean_all` is now log1p CP10K on the **true** library
   (`obs.total_counts`, median 4,127), not on the sum over the 10,000 HVG columns (median
   2,186). The HVG-capture ratio differs between a clean T cell and a myeloid doublet, which is
   the confound under test; `mean_all_hvg` keeps the old quantity so the change is auditable.

In [ ]:
# ============================================================================
# §1  Load the rest of the QC build
# ============================================================================
meta = TC.filter_long(pd.read_parquet(TARGETS / f"{TISSUE}_target_meta.parquet"), don)
man = json.loads((TARGETS / f"{TISSUE}_target_manifest.json").read_text())
cl_path = TARGETS / f"{TISSUE}_target_cluster.parquet"
cluster_long = TC.filter_long(pd.read_parquet(cl_path), don) if cl_path.exists() else None
qc_path = TARGETS / f"{TISSUE}_target_qc_cell.parquet"
qc_cell = pd.read_parquet(qc_path) if qc_path.exists() else None

assert man.get("builder") == "run_target_screen_qc.py", (
    "these tables came from the v1 builder, which has no robust statistics, no clean recompute "
    "and no lineage controls -- run jobs/run_target_screen_qc.sh --tissue blood first. "
    f"manifest says: {man.get('builder', 'run_target_screen.py')}")

print(f"source {Path(man['source']).name} | call {man['malignancy_call']} | "
      f"{man['n_genes_in_object']:,} genes in object -> "
      f"{man['gene_selection'].get('n_total', man['gene_selection'].get('n_screened')):,} "
      f"screened (+{man['gene_selection'].get('n_lineage_added', 0)} lineage controls, "
      f"+{man['gene_selection'].get('n_traffic_added', 0)} trafficking)")
if man["n_genes_in_object"] < 20000:
    print("*** HVG subset: absent is NOT unexpressed. Quote this next to every blood result. ***")
print("library:", man.get("library"))
print("clean rule:", json.dumps(man.get("clean_rule", {}), default=str))
print(f"depth matching: {man.get('ds_depth'):,} UMI on {man.get('frac_cells_ds', 0):.1%} "
      "of cells")

arms = (meta.groupby("arm").agg(donors=("donor", "nunique"), cells=("n_cells", "sum"),
                                studies=("study", "nunique"))
        .sort_values("cells", ascending=False))
print("\narms, independent cohort:")
print(arms.to_string())

paired = meta[meta["arm"].isin([MALIG, REACT])].groupby("donor")["arm"].nunique().eq(2)
PAIRED_DONORS = sorted(paired.index[paired])
print(f"\ndonors with both arms: {len(PAIRED_DONORS)}")
DONOR_STUDY = dict(zip(meta["donor"], meta["study"]))

print("\ncontamination load per arm (median over donors) -- the asymmetry the artifact needs:")
print(meta.groupby("arm")[["frac_lineage_pos", "mean_myeloid_score", "median_total_counts",
                           "median_doublet_score"]].median().round(4).to_string())

## §2 — Is the ranking contamination? A test that is allowed to fail

The charge against the first ranking is specific, so the test can be too.

**The mechanism.** Blood in SS is ~92% malignant CD4. Malignant×monocyte doublets therefore
outnumber reactive×monocyte doublets by roughly that ratio, so every minor population in the
sample preferentially contaminates the malignant arm — in every donor, which is why a technical
effect of this kind reproduces *better* than real inter-patient biology and why `frac_donors_up`
came out inverted against plausibility. Ambient RNA is shared by both arms and cannot do this.

**Six tests**, each with its own failure mode:

| test | what fires it |
|---|---|
| `fails_trim` | the effect collapses when the top 1% of cells is trimmed — carried by a handful of cells, not by the population |
| `fails_concentration` | more of the arm's counts sit in its top 1% of cells than for other genes *detected at the same rate* |
| `fails_lineage` | per-cell expression tracks the myeloid module score **inside** the malignant arm (the test is one module here and three in skin) |
| `fails_umi` | it tracks per-cell library size, or detection is confined to the top-UMI decile, beyond what its prevalence implies |
| `fails_clean` | it does not survive dropping lineage-positive and high-doublet-score cells |
| `fails_burden` | its per-donor effect grows with that donor's malignant:reactive abundance ratio |

Two failures ⇒ `artifact`, one ⇒ `suspect`. Only up-in-malignant genes can be flagged;
contamination entering the malignant arm cannot push a gene down.

**The null, and the escape hatch.** `LYZ`, `S100A8`, `S100A9`, `CD14`, `VCAN`, `FCN1`, `CST3`,
`LST1`, `AIF1`, `TYROBP`, `FCER1G`, `CD68`, `MNDA`, `PPBP`, `PF4`, `HBB`, `MS4A1`, `CD79A` are
not CD4 T-cell transcripts, so whatever they score *is* what contamination scores in this
cohort, and every threshold is read off them rather than chosen. That also makes the section
refutable: **if those genes are not up in the malignant arm, the mechanism is not operating
here, and §2b excludes nothing.** The calibration verdict prints before any flag is assigned —
read it first, because it decides whether the rest of the section means anything.

**Why the statistics are prevalence-matched.** `top1_share`, the top-UMI-decile odds ratio and
the myeloid correlation are all functions of how often a gene is detected at all: a gene seen
in two cells necessarily concentrates, and a gene seen in 80% of cells necessarily rises with
depth because detection saturates. Compared raw, the band would condemn CD7 and acquit a gene
seen once. Each is taken as a residual against genes of the same detection rate.

**§2d adds the two artifacts that are not contamination** — the malignancy call being a cluster
label while the comparison is between those same clusters, and the effect living in one or two
clusters rather than in the clone. The control for the first is the whole screen rebuilt with
the arms taken from the TCR clone, which is measured independently of the transcriptome.

In [ ]:
# ============================================================================
# §2a  The six artifact statistics, and the empirical contamination null
# ============================================================================
# The same paired test, four times, on four location estimators. If an effect is carried by a
# handful of cells it survives only the first.
sel = TS.paired_selectivity(long, MALIG, REACT, value="mean_all", min_donors=MIN_DONORS)
sel_trim = TS.paired_selectivity(long, MALIG, REACT, value="trim1_mean", min_donors=MIN_DONORS)
sel_wins = TS.paired_selectivity(long, MALIG, REACT, value="wins90_mean", min_donors=MIN_DONORS)
sel_clean = TS.paired_selectivity(long, MALIG, REACT, value="mean_all_clean",
                                  min_donors=MIN_DONORS)
sel_ds = TS.paired_selectivity(long, MALIG, REACT, value="mean_all_ds", min_donors=MIN_DONORS)
blk = TS.blocked_selectivity(long, MALIG, REACT, block=DONOR_STUDY, value="mean_all")
sel = sel.merge(blk, on="gene", how="left")

conc = TQ.concentration(long, MALIG)                       # top1/top5 share, detection
brn = TQ.burden_scaling(long, bur, MALIG, REACT)           # does the effect scale with burden
confine = (TQ.cluster_confinement(cluster_long, MALIG)
           if cluster_long is not None else None)

ev = TQ.assemble(sel, sel_trim, sel_clean, qc_cell, brn, conc, confine)
for name, s in (("wins", sel_wins), ("ds", sel_ds)):
    ev = ev.merge(s[["gene", "log2fc_paired"]].rename(
        columns={"log2fc_paired": f"log2fc_{name}"}), on="gene", how="left")
    ev[f"retained_{name}"] = TQ._ratio(ev[f"log2fc_{name}"], ev["log2fc_paired"])
ev = TQ.prevalence_residual(ev)

print(f"{len(sel):,} genes tested paired on {len(PAIRED_DONORS)} donors | "
      f"{int((sel['q'] < 0.05).sum()):,} at q < 0.05")

print("\n" + "=" * 78)
print("THE NULL: what pure contamination scores in this cohort")
print("=" * 78)
ok, msg = TQ.band_is_calibrated(ev, None)
print(msg)
band = TQ.contamination_band(ev)
print(f"\nlineage controls found: {band.attrs['n_genes']} "
      f"({', '.join(band.attrs['genes_found'])})")
print("\nper-gene, in full:")
_bcols = [c for c in ("gene", "log2fc_paired", "frac_donors_up", "median_pct_expr",
                      "retained_trim", "retained_clean", "top1_share_excess",
                      "r_myeloid_excess", "or_topdecile_excess", "rho_burden")
          if c in ev.columns]
print(ev[ev["gene"].isin(TQ.BAND_GENES)][_bcols]
      .sort_values("log2fc_paired", ascending=False)
      .to_string(index=False, float_format=lambda v: f"{v:.4g}"))
print("\nquantiles of the band (thresholds are read from the p90 row):")
print(band.round(4).to_string())

In [ ]:
# ============================================================================
# §2b  The verdict -- which is only allowed to exclude anything if the null is calibrated
# ============================================================================
out = TQ.artifact_evidence(ev, band)
if not ok:
    # The lineage controls are the whole basis for calling a gene contamination. If they are not
    # up in the malignant arm, the mechanism is absent from this cohort and the six tests are
    # measuring something else. Nothing is excluded; the evidence is still printed, because the
    # columns are informative about every gene whether or not contamination is the explanation.
    out["artifact_flag"] = "clean"
    out["fail_reasons"] = "- (null not calibrated; no exclusion applied)"
    print("*" * 78)
    print("NO GENE IS EXCLUDED ON CONTAMINATION GROUNDS.")
    print(msg)
    print("*" * 78)

ev = out
print()
print(TQ.verdict_text(ev))

print("\n" + "=" * 78)
print("the six tests, per gene, for every gene with a positive effect at q < 0.05")
print("=" * 78)
_full = ev[(ev["q"] < 0.05) & (ev["log2fc_paired"] > 0)].sort_values("log2fc_paired",
                                                                    ascending=False)
_fc = [c for c in ("gene", "log2fc_paired", "frac_donors_up", "median_pct_expr",
                   "retained_trim", "retained_wins", "retained_clean", "retained_ds",
                   "top1_share_excess", "r_myeloid_excess", "r_umi_excess",
                   "or_topdecile_excess", "rho_burden", "top_cluster_share",
                   "n_fails", "fail_reasons") if c in ev.columns]
print(_full[_fc].to_string(index=False, float_format=lambda v: f"{v:.4g}"))

ev.to_csv(ST.tab("artifact_evidence"), index=False)
band.to_csv(ST.tab("contamination_band"))
print("\nwrote", ST.tab("artifact_evidence"), "and", ST.tab("contamination_band"))
print("\nColumn key:  retained_* = effect kept after trimming the top 1% of cells / "
      "winsorising at the 90th percentile / dropping lineage-positive and high-doublet cells / "
      "binomially thinning to a common depth. 1.0 = unchanged, 0 = gone. *_excess = the "
      "statistic minus the cohort median for genes detected at the same rate. rho_burden = "
      "Spearman of the per-donor effect against that donor's malignant:reactive ratio.")

In [ ]:
# ============================================================================
# §2c FX  The null, and where the disputed genes actually sit
# ============================================================================
SETS = {"lineage controls\n(pure contamination)": TQ.BAND_GENES,
        "all screened genes": sorted(set(ev["gene"]) - set(TQ.BAND_GENES)),
        "disputed genes": TQ.SUSPECTS,
        "validated antigens": ST.POSITIVE_CONTROLS}
fig, axes = plt.subplots(1, 2, figsize=(9.2, 3.4),
                         gridspec_kw={"width_ratios": [1.05, 1.0]})

ax = axes[0]
rng = np.random.default_rng(0)
for i, (lab, gs) in enumerate(SETS.items()):
    v = ev.loc[ev["gene"].isin(gs), "log2fc_paired"].dropna()
    col = [TF.DOWN, "#bdbdbd", "#2e86c1", TF.UP][i]
    ax.scatter(np.full(len(v), i) + rng.normal(0, .07, len(v)), v, s=7 if i == 1 else 18,
               color=col, alpha=.35 if i == 1 else .9, linewidths=0)
    if len(v):
        ax.hlines(np.median(v), i - .25, i + .25, color="#111111", lw=1.4, zorder=5)
ax.axhline(0, lw=.5, c="k")
ax.set_xticks(range(len(SETS)))
ax.set_xticklabels(SETS, fontsize=6)
ax.set_ylabel("log2FC, malignant vs reactive CD4")
ax.set_title("Contamination null vs candidates", loc="left", fontsize=7.5)
TF._clean(ax)

# Right: the three-arm view. Contamination that rides into the malignant arm has no reason to
# spare healthy CD4 -- healthy blood carries monocytes too. A transcript that is absent in
# healthy and reactive CD4 and present in the clone is tumour-restricted, whatever one thinks
# of its gene name.
ax = axes[1]
show = TQ.SUSPECTS + ST.POSITIVE_CONTROLS + ["CD7"]
det = (long[long["gene"].isin(show) & long["arm"].isin([HC_CD4, REACT, MALIG])]
       .pivot_table(index="gene", columns="arm", values="pct_expr", aggfunc="median")
       .reindex([g for g in show if g in set(long["gene"])])
       .sort_values(MALIG))
for j, (arm, col, mk) in enumerate(((HC_CD4, TF.COMPARATOR, "o"), (REACT, TF.REACT, "o"),
                                    (MALIG, TF.MALIG, "o"))):
    if arm in det:
        ax.scatter(det[arm], np.arange(len(det)) + (j - 1) * 0.2, s=22, color=col,
                   marker=mk, linewidths=0, label=arm.replace("CD4_", ""))
for y, (_, r_) in enumerate(det.iterrows()):
    ax.plot([r_.min(), r_.max()], [y, y], color="#dddddd", lw=0.8, zorder=0)
ax.set_yticks(range(len(det)))
ax.set_yticklabels([TF.it(g) for g in det.index], fontsize=6.5)
ax.set_xlabel("median fraction of a donor's cells detected")
ax.set_xscale("symlog", linthresh=1e-3)
ax.set_xlim(left=0)
ax.set_ylim(-0.7, len(det) - 0.3)
ax.legend(fontsize=5.5, frameon=False, loc="upper left", title="CD4 arm",
          title_fontsize=5.5)
ax.set_title("Detection across CD4 arms", loc="left", fontsize=7.5)
TF._clean(ax)

fig.tight_layout()
TF.save(fig, "FX_artifact_null", caption_text=(
    "Left: paired log2FC by gene set. Right: median detection per CD4 arm, symlog x."))
TF.save_source(det.reset_index(), "FX_three_arm_detection_source")
TF.logfig("FX_artifact_null", "contamination null vs candidates; three-arm detection",
          len(ev))

print("median detection by arm, the genes in question (and the controls):")
print(det.round(4).to_string())

In [ ]:
# ============================================================================
# §2d  Two more ways the effect could be an artifact, and neither is contamination
# ============================================================================
# (1) CLUSTER CIRCULARITY. The blood malignancy call is `mal_cluster` -- a *cluster* label. The
#     arms are therefore groups of Leiden clusters, and the comparison is between the same
#     clusters whose expression defined them. A gene confined to one or two malignant clusters
#     would be a cluster marker rediscovered as an antigen.
if confine is not None:
    print("share of the malignant arm's signal held by its single strongest cluster "
          f"({int(confine['n_clusters'].median())} clusters):")
    _q = confine["top_cluster_share"].describe(percentiles=[.5, .9, .99]).round(4)
    print("  all screened genes:", _q.to_dict())
    print("\n  the genes in question:")
    print(confine[confine["gene"].isin(TQ.SUSPECTS + ST.POSITIVE_CONTROLS)]
          .sort_values("top_cluster_share", ascending=False)
          .to_string(index=False, float_format=lambda v: f"{v:.4g}"))
    print("\n  A gene spread evenly over N clusters scores 1/N. Confinement would show as a "
          "share far above that floor.")

# (2) THE CALL ITSELF. `mal_tcr` assigns malignancy from the expanded TCR clone, which is
#     measured independently of the transcriptome. If an effect holds under that call it cannot
#     be an artifact of how the clusters were drawn. Built by:
#         bash jobs/run_target_screen_qc.sh --tissue blood --call mal_tcr --no-cluster \
#              --out-dir data/targets/by_call_mal_tcr
TCR_DIR = TARGETS / "by_call_mal_tcr"
_keep = ["gene", "log2fc_paired", "frac_donors_up", "n_donors_paired"]
if (TCR_DIR / f"{TISSUE}_target_long.parquet").exists():
    long_tcr = TC.filter_long(pd.read_parquet(TCR_DIR / f"{TISSUE}_target_long.parquet"), don)
    sel_tcr = TS.paired_selectivity(long_tcr, MALIG, REACT, value="mean_all",
                                    min_donors=MIN_DONORS)
    both_calls = sel[_keep].merge(sel_tcr[_keep], on="gene", suffixes=("_cluster", "_tcr"))
    _r = stats.spearmanr(both_calls["log2fc_paired_cluster"], both_calls["log2fc_paired_tcr"])
    print("\n" + "=" * 78)
    print(f"the same screen with the arms built from the TCR clone instead of the cluster: "
          f"rho = {_r.statistic:.2f} over {len(both_calls):,} genes "
          f"({both_calls['n_donors_paired_tcr'].median():.0f} paired donors vs "
          f"{both_calls['n_donors_paired_cluster'].median():.0f})")
    print("=" * 78)
    print(both_calls[both_calls["gene"].isin(TQ.SUSPECTS + ST.POSITIVE_CONTROLS
                                             + ST.LOSS_MARKERS)]
          .sort_values("log2fc_paired_cluster", ascending=False)
          .to_string(index=False, float_format=lambda v: f"{v:.3g}"))
    both_calls.to_csv(ST.tab("call_sensitivity"), index=False)
    print("\nAn effect that is an artifact of the clustering cannot survive a call made from "
          "the TCR. One that does survive is a property of the clone.")
    print("wrote", ST.tab("call_sensitivity"))
else:
    print(f"\n{TCR_DIR.name} not built -- the cluster-circularity control is MISSING. "
          "Run: bash jobs/run_target_screen_qc.sh --tissue blood --call mal_tcr --no-cluster "
          f"--out-dir {TCR_DIR}")

## §3 — Malignant vs reactive CD4, on what survived

The comparison the whole screen exists for, now on an independent cohort and on genes that
passed §2. Five figures, in the style of `28_tme_degradome_mc`, because a cohort median hides
the thing that decides whether a target is usable: **what it does inside each patient.**

| figure | what it shows |
|---|---|
| **FA** | the leading antigens, one grey line per donor joining its reactive and malignant means |
| **FB** | the antigens whose answer is already known — read this one first |
| **FC** | the ranked candidate list as a dumbbell, with prevalence and off-tumour beside each row |
| **FD** | the donor × gene effect matrix, donors ordered by malignant burden |
| **FE** | the same comparison on *detection rate*, which is what a CAR is dosed against |

The star above each gene is the BH-adjusted paired Wilcoxon; `n↑/n↓` underneath it is the donor
vote, and the two percentages under each gene name are the median share of cells detected in
the malignant and reactive arms. Titles say what the panel is; the finding is in the table
printed under it.

FD is the figure to argue with. Donors run left to right by malignant share of their CD4
compartment, so a row that darkens towards the right is a row whose effect tracks arm
composition — the pattern §2 tests for formally with `fails_burden`, shown here so it can also
be seen.

In [ ]:
# ============================================================================
# §3  Malignant vs reactive CD4, on the genes that survived §2
# ============================================================================
prev = TS.prevalence(long, MALIG, detect_floor=DETECT_FLOOR)
off = TS.off_tumour(long, target_arms=[MALIG], healthy_t_arms=[HC_CD4], value="pct_expr")
off = off.rename(columns={"worst_offtumour": "worst_T_arm_pct",
                          "worst_offtumour_arm": "worst_T_arm"})
esc = TS.escape_risk(long, MALIG, cluster_long=cluster_long)

core = (sel.merge(prev, on="gene", how="left")
           .merge(off, on="gene", how="left")
           .merge(ev[["gene", "artifact_flag", "n_fails", "fail_reasons",
                      "retained_trim", "retained_clean", "retained_ds", "rho_burden"]],
                  on="gene", how="left"))
core = core.merge(ST.annotate(core["gene"])[["gene", "cd_name", "panel_class", "stage"]],
                  on="gene", how="left")

GATES = ((core["artifact_flag"].eq("clean")) & (core["q"] < 0.05)
         & (core["log2fc_paired"] > 0.5) & (core["frac_donors_positive"] >= 0.5)
         & (~core["gene"].isin(ST.LOSS_MARKERS)) & (~core["gene"].isin(TQ.BAND_GENES)))
up = core[GATES].sort_values("log2fc_paired", ascending=False)
SHORT_GENES = up["gene"].tolist()[:30]
TOP_GENES = SHORT_GENES[:12]

CAPTION = (f"Paired Wilcoxon within donor, BH. {len(PAIRED_DONORS)} donors, "
           f"{meta['study'].nunique()} studies. log1p CP10K.")
LINE_CAPTION = CAPTION + " Lines: red = up in malignant, blue = down."

print(f"{len(up)} genes pass every gate "
      "(clean, q<0.05, log2FC>0.5, detected in >=50% of donors):\n")
_c = ["gene", "log2fc_paired", "frac_donors_up", "frac_donors_positive", "median_pct_expr",
      "q", "z_blocked", "retained_trim", "retained_clean", "retained_ds", "rho_burden",
      "worst_T_arm", "cd_name", "stage"]
print(up[[c for c in _c if c in up]].to_string(index=False,
                                               float_format=lambda v: f"{v:.4g}"))

print("\nloss markers -- CD7 and DPP4/CD26 loss is the Sezary immunophenotype, and is the "
      "sanity check that the cleaning did not flatten real biology:")
print(core[core["gene"].isin(ST.LOSS_MARKERS)][["gene", "log2fc_paired", "frac_donors_up",
                                                "q", "artifact_flag"]]
      .sort_values("log2fc_paired").to_string(index=False,
                                              float_format=lambda v: f"{v:.4g}"))

In [ ]:
# ============================================================================
# §3 FA  The leading surviving antigens, one line per donor
# ============================================================================
recA = TF.t_paired(long, TOP_GENES, MALIG, REACT, value="mean_all", min_donors=MIN_DONORS)
recA = recA.merge(ST.annotate(recA["gene"])[["gene", "cd_name", "stage"]], on="gene", how="left")

fig, ax = plt.subplots(figsize=(0.82 * len(recA) + 1.8, 3.6))
TF.paired_slopes(ax, recA)
ax.set_title(f"Top antigens, malignant vs reactive CD4 (n={len(PAIRED_DONORS)})",
             loc="left", fontsize=7.5)
TF.save(fig, "FA_top_candidates_paired", caption_text=LINE_CAPTION)
TF.save_source(recA, "FA_top_candidates_paired_source")
TF.logfig("FA_top_candidates_paired", "leading clean antigens, paired per donor", len(recA))

TF.show_tests(recA, cols=TF.SHOW_COLS + ["cd_name", "stage"])

In [ ]:
# ============================================================================
# §3 FB  Calibration: the antigens whose answer is already known
# ============================================================================
# Read this panel before FA. KIR3DL2 and CCR4 are the positive controls -- CCR4 is approved in
# CTCL, KIR3DL2 is lacutamab's target. CD7 and DPP4/CD26 loss IS the Sezary immunophenotype, so
# they must come out strongly negative. If this panel does not reproduce, nothing in FA is
# evidence of anything, and the cleaning in §2 is the first suspect.
CTRL_GENES = [g for g in (ST.POSITIVE_CONTROLS + ST.LOSS_MARKERS)
              if g in set(long["gene"])]
recB = TF.t_paired(long, CTRL_GENES, MALIG, REACT, value="mean_all", min_donors=MIN_DONORS)
recB = recB.merge(ev[["gene", "artifact_flag"]], on="gene", how="left")

fig, ax = plt.subplots(figsize=(1.0 * len(recB) + 1.6, 3.4))
TF.paired_slopes(ax, recB)
ax.set_title("Validated antigens and Sézary loss markers", loc="left", fontsize=7.5)
TF.save(fig, "FB_controls_paired", caption_text=LINE_CAPTION)
TF.save_source(recB, "FB_controls_paired_source")
TF.logfig("FB_controls_paired", "positive controls and loss markers, paired", len(recB))

TF.show_tests(recB, cols=TF.SHOW_COLS + ["artifact_flag"])
_missing = sorted(set(ST.POSITIVE_CONTROLS + ST.LOSS_MARKERS) - set(long["gene"]))
if _missing:
    print(f"\nnot testable, absent from the 10k HVG var: {_missing}")
    print("TNFRSF8 is CD30, the brentuximab target, and CD26 is an alias of DPP4 which is "
          "present -- so the real loss is CD30 alone.")

In [ ]:
# ============================================================================
# §3 FC  The surviving candidates as one ranked panel
# ============================================================================
recC = TF.t_paired(long, SHORT_GENES, MALIG, REACT, value="mean_all", min_donors=MIN_DONORS)
recC = recC.merge(ST.annotate(recC["gene"])[["gene", "cd_name", "stage"]], on="gene", how="left")
recC = recC.merge(core[["gene", "frac_donors_positive", "median_pct_expr", "worst_T_arm",
                         "worst_T_arm_pct"]], on="gene", how="left")
recC = recC.merge(prev[["gene", "n_donors"]].rename(columns={"n_donors": "n_donors_malig"}),
                  on="gene", how="left")
recC["ylabel"] = [f"{TF.it(r['gene'])}" + (f" · {r['cd_name']}" if r["cd_name"] else "")
                  for _, r in recC.iterrows()]
recC = recC.sort_values("log2fc", ascending=False)

fig, _ = TF.shortlist_panel(recC, floor=DETECT_FLOOR,
                            title="Shortlist: effect, patients positive, off-tumour burden")
TF.save(fig, "FC_shortlist_dumbbell", caption_text=CAPTION)
TF.save_source(recC, "FC_shortlist_dumbbell_source")
TF.logfig("FC_shortlist_dumbbell", "ranked clean candidates, both arms", len(recC))

TF.show_tests(recC, cols=TF.SHOW_COLS + ["cd_name", "stage", "frac_donors_positive",
                                         "median_pct_expr", "worst_T_arm", "worst_T_arm_pct"])

In [ ]:
# ============================================================================
# §3 FD  Leading markers inside each sample: the donor x gene effect matrix
# ============================================================================
M = TH.lfc_matrix(long, MALIG, REACT, genes=TOP_GENES, min_donors=MIN_DONORS)
M = M.loc[[g for g in TOP_GENES if g in M.index]]
# donors ordered by their own malignant burden, so the eye can check the one thing §2 warned
# about: an effect that grows left to right is an effect that grows with arm composition.
_b = bur["burden"].reindex(M.columns).dropna().sort_values()
M = M[list(_b.index)]

fig, ax = plt.subplots(figsize=(0.13 * M.shape[1] + 3.2, 0.19 * len(M) + 1.8))
TF.donor_gene_heat(ax, M, stars_by=dict(zip(core["gene"], core["q"])))
ax.set_xlabel(f"donor, ordered by malignant share of CD4 "
              f"({_b.iloc[0]:.0%} → {_b.iloc[-1]:.0%})")
ax.set_title("Per-donor log2FC, malignant vs reactive", loc="left", fontsize=7.5)
TF.save(fig, "FD_donor_gene_heat", caption_text=CAPTION)
TF.save_source(M.reset_index(), "FD_donor_gene_heat_source")
TF.logfig("FD_donor_gene_heat", "per-donor log2FC matrix, donors by burden", M.size)

print("per-donor log2FC, complete matrix (genes x donors):")
print(M.round(3).to_string())
print("\nper-gene summary of that matrix:")
_sum = pd.DataFrame({
    "n_donors": M.notna().sum(1),
    "median": M.median(1), "iqr": M.quantile(.75, axis=1) - M.quantile(.25, axis=1),
    "min": M.min(1), "max": M.max(1),
    "frac_up": (M > 0).sum(1) / M.notna().sum(1),
    "frac_up_1": (M > 1).sum(1) / M.notna().sum(1),
}).sort_values("median", ascending=False)
print(_sum.to_string(float_format=lambda v: f"{v:.3g}"))
print("\n`frac_up_1` -- donors where the antigen is at least doubled -- is the column that "
      "separates a cohort-wide antigen from one that is strong in a subset. 44 takes that "
      "apart formally.")

In [ ]:
# ============================================================================
# §3 FE  Detection, not expression: the axis a CAR is actually dosed against
# ============================================================================
recE = TF.t_paired(long, TOP_GENES, MALIG, REACT, value="pct_expr", min_donors=MIN_DONORS)
fig, ax = plt.subplots(figsize=(10.2, 3.4))
TF.paired_slopes(ax, recE, ylabel="fraction of a donor's cells detected", show_detection=False)
ax.set_title("Fraction of CD4 cells detected, per donor", loc="left", fontsize=7.5)
TF.save(fig, "FE_detection_paired", caption_text=LINE_CAPTION)
TF.save_source(recE, "FE_detection_paired_source")
TF.logfig("FE_detection_paired", "paired detection rate per donor", len(recE))

print("Detection is the quantity that decides whether a CAR sees the tumour; expression is the "
      "quantity the ranking is sorted on. They are not the same ordering, and where they "
      "disagree the detection column wins for a cell-therapy target.\n")
TF.show_tests(recE)

print("\n" + "=" * 78)
print("eligibility: in what fraction of patients would at least X% of the tumour be positive")
print("=" * 78)
elig = TH.eligibility(long, MALIG, genes=TOP_GENES, donors=PAIRED_DONORS)
print(elig.to_string(index=False, float_format=lambda v: f"{v:.3g}"))
elig.to_csv(ST.tab("eligibility_blood"), index=False)
print("\nwrote", ST.tab("eligibility_blood"))

## §4 — On-target off-tumour in blood

Blood is the compartment where an off-tumour hit is most consequential: a CAR that also binds
monocytes, NK cells or HSPCs causes cytopenia, not a rash. The non-T compartments come from
`31b_subtype_reannotation`; OneK1K (981 healthy donors) is the healthy-PBMC reference.

In [ ]:
# ============================================================================
# §4  Off-tumour
# ============================================================================
print("worst non-target T arm, for the surviving candidates:")
_o = (core[core["gene"].isin(SHORT_GENES)]
      [["gene", "cd_name", "worst_T_arm", "worst_T_arm_pct", "healthy_t_burden",
        "median_pct_expr", "frac_donors_positive"]]
      .sort_values("worst_T_arm_pct", ascending=False))
print(_o.to_string(index=False, float_format=lambda v: f"{v:.3g}"))
print("\n`healthy_t_burden` is the aplasia cost: the share of HEALTHY CD4 cells that carry the "
      f"antigen, now measured on {meta.loc[meta['arm'].eq(HC_CD4), 'donor'].nunique()} healthy "
      "donors rather than 7, because the five geskin donors that the atlas labelled SS are "
      "healthy and have been moved into this arm.")

# What this axis does NOT cover, stated as a limit rather than a plan. The screen input is
# blood_T_annotated.h5ad, the T/NK compartment only; the non-T cells live in the joint object
# and would need their own pass of jobs/run_target_screen_qc.py to enter `long`.
SUBTYPES = NB_DIR / "data" / "atlas_joint" / "blood_subtypes.csv"
if SUBTYPES.exists():
    sub = pd.read_csv(SUBTYPES, usecols=["subtype_coarse"])
    print(f"\n{SUBTYPES.name}: {len(sub):,} non-T blood cells annotated by 31b "
          f"({sub['subtype_coarse'].nunique()} coarse types) -- but NOT in this axis.")
else:
    print(f"\n{SUBTYPES.name} NOT found: run 31b_subtype_reannotation first.")
print("*** The off-tumour axis above covers T and NK arms ONLY. Monocytes, DC, B, plasma, "
      "platelets and HSPC are not in `long`, so a myeloid or HSPC hit -- the kind that causes "
      "cytopenia, the dose-limiting toxicity for a blood-directed CAR -- is INVISIBLE here. "
      "Quote every blood candidate with this sentence attached. ***")

ONEK1K = NB_DIR / "data" / "external_ref" / "onek1k.h5ad"
print(f"\nhealthy PBMC reference {'present' if ONEK1K.exists() else 'ABSENT'}: {ONEK1K.name} "
      "(not yet folded in either)")

In [ ]:
# ============================================================================
# §5-6  Surface plausibility (trafficking context) + escape risk
# ============================================================================
tr = long[long["gene"].isin(ST.TRAFFIC_GENES)].copy()
tr["module"] = tr["gene"].map(lambda g: ST.GENE_TO_TRAFFIC.get(g, [""])[0])
piv = tr.groupby(["arm", "donor", "module"])["mean_all"].mean().reset_index()
piv["z"] = piv.groupby("module")["mean_all"].transform(lambda s: (s - s.mean()) / (s.std() or 1))
arm_mod = piv.pivot_table(index="module", columns="arm", values="z")
print("trafficking module z by arm -- the capacity to put a protein on the surface and keep "
      "it there:")
print(arm_mod[[c for c in (MALIG, REACT, HC_CD4) if c in arm_mod]].round(2).to_string())
print("\nThis is context, not a correction factor. 45 asks the sharper version of the same "
      "question on measured protein: is a marker's surface level better explained by these "
      "modules than by its own transcript.")

print("\nescape risk computed on",
      "32b clusters" if cluster_long is not None else "donors only")
print(esc[esc["gene"].isin(SHORT_GENES)]
      .sort_values("between_donor_sd", ascending=False)
      .to_string(index=False, float_format=lambda v: f"{v:.3g}"))
print("\n`between_donor_sd` is the spread of detection across patients and "
      "`within_donor_sd` the spread across a patient's own clusters -- a high second column is "
      "the antigen-negative subclone a single-target agent would select for. 44 takes the "
      "between-patient term apart properly.")

In [ ]:
# ============================================================================
# §7  Assemble, check the controls, shortlist
# ============================================================================
surf = ST.load_surfaceome(strict=False)
rank = TS.rank_table(prev, sel, off, esc, ST.annotate, surfaceome=surf)
rank = rank.merge(ev[["gene", "artifact_flag", "n_fails", "fail_reasons", "retained_trim",
                      "retained_clean", "retained_ds", "rho_burden", "top1_share"]],
                  on="gene", how="left")
if "top_cluster_share" in ev:
    rank = rank.merge(ev[["gene", "top_cluster_share"]], on="gene", how="left")

hit, missing, n_tested = TS.control_check(rank, ST.POSITIVE_CONTROLS)
print(f"positive controls of {n_tested:,} ranked genes:")
print(hit.to_string(index=False))
if missing:
    print(f"*** not testable: {missing} -- dropped by HVG selection, so the calibration runs "
          f"on {len(hit)} of {len(ST.POSITIVE_CONTROLS)} controls ***")

short = rank[rank["gene"].isin(SHORT_GENES)].copy()
show = ["gene", "cd_name", "frac_donors_positive", "median_pct_expr", "log2fc_paired", "q",
        "worst_T_arm", "healthy_t_burden", "between_donor_sd", "stage", "n_evidence",
        "surface_confident", "n_tm", "surfy_class", "artifact_flag"]
print(f"\n{len(short)} candidates pass the gates:")
print(short[[c for c in show if c in short]]
      .to_string(index=False, float_format=lambda v: f"{v:.4g}"))

print("\n" + "=" * 78)
print("PLAUSIBILITY, which the statistics cannot decide")
print("=" * 78)
print("`surface_confident` / `n_tm` / `surfy_class` come from CSPA + Surfy, not from this "
      "cohort. A transcript can be reproducibly, tumour-restrictedly up and still be a hopeless "
      "antibody or CAR target because the protein is cytoplasmic, secreted, or expressed on a "
      "tissue this atlas cannot see. Genes with no surfaceome evidence are the ones to argue "
      "about before anyone orders a reagent:")
_noev = short[short["surface_confident"].isna() | ~short["surface_confident"].astype(bool)]
print(_noev[["gene", "cd_name", "log2fc_paired", "median_pct_expr", "n_evidence",
             "surfy_class", "topology"]].to_string(index=False,
                                                   float_format=lambda v: f"{v:.3g}")
      if len(_noev) else "  every shortlisted gene has confident surfaceome evidence")

rank.to_csv(ST.tab("candidates_blood"), index=False)
short.to_csv(ST.tab("shortlist_blood"), index=False)
print("\nwrote", ST.tab("candidates_blood"), "and", ST.tab("shortlist_blood"))

## §8 — Skin ∩ blood

A target that separates malignant from reactive in **both** compartments is the headline result:
MF is a skin disease that becomes a blood disease, and an agent that works in one compartment
and not the other leaves the reservoir that repopulates it.

Discordance is informative too — a skin-only hit is a candidate for intralesional or topical
delivery, a blood-only hit for a systemic agent in SS.

> **Both sides have now had the same treatment.** 42 was rebuilt on this notebook's template:
> independent donors only, diagnosis from `clin_dx`, true-library normalisation, and the same
> six artifact tests against a skin-specific contamination null (keratin, collagen, melanocyte
> and myeloid transcripts in place of the monocyte and platelet ones). So the two columns below
> are comparable in a way they were not before.
>
> Two asymmetries remain and neither is fixable here. The blood object is a 10,000-gene HVG
> subset and the skin object is not, so a gene can be absent from the blood column for a reason
> that has nothing to do with biology. And the skin screen can see the whole tissue — its
> off-tumour axis covers keratinocytes, fibroblasts and myeloid cells — while the blood screen
> sees only the T and NK arms.

In [ ]:
# ============================================================================
# §8  Skin and blood side by side
# ============================================================================
skin_tab = ST.TAB_DIR / "tgt42_candidates_skin.csv"
if not skin_tab.exists():
    print(f"{skin_tab.name} not found -- run 42 first")
else:
    sk = pd.read_csv(skin_tab)
    both = sk.merge(rank, on="gene", suffixes=("_skin", "_blood"))
    both = both.dropna(subset=["log2fc_paired_skin", "log2fc_paired_blood"])
    print(f"{len(both):,} genes screened in both compartments "
          "(both sides cohort-fixed and artifact-tested; see the note above for what still "
          "differs)")

    sig = both[(both["q_skin"] < 0.05) & (both["q_blood"] < 0.05)]
    up_both = sig[(sig["log2fc_paired_skin"] > 0.5) & (sig["log2fc_paired_blood"] > 0.5)]
    print(f"\nselective in BOTH compartments: {len(up_both)}")
    cols = ["gene", "cd_name_skin", "log2fc_paired_skin", "log2fc_paired_blood",
            "frac_donors_positive_skin", "frac_donors_positive_blood", "stage_skin"]
    print(up_both[[c for c in cols if c in up_both]]
          .sort_values("log2fc_paired_blood", ascending=False)
          .to_string(index=False, float_format=lambda v: f"{v:.4g}"))

    fig, ax = plt.subplots(figsize=(5, 3.4))
    ax.axhline(0, lw=.5, c="k"); ax.axvline(0, lw=.5, c="k")
    ax.scatter(both["log2fc_paired_skin"], both["log2fc_paired_blood"], s=4,
               c="#d5d8dc", linewidths=0)
    k = both["gene"].isin(up_both["gene"])
    ax.scatter(both.loc[k, "log2fc_paired_skin"], both.loc[k, "log2fc_paired_blood"], s=10,
               c="#2e86c1", linewidths=0)
    kc = both["gene"].isin(ST.POSITIVE_CONTROLS + TQ.SUSPECTS)
    ax.scatter(both.loc[kc, "log2fc_paired_skin"], both.loc[kc, "log2fc_paired_blood"], s=26,
               c=TF.DOWN, linewidths=0)
    for _, r_ in both[kc].iterrows():
        ax.annotate(r_["gene"], (r_["log2fc_paired_skin"], r_["log2fc_paired_blood"]),
                    fontsize=5.5, xytext=(3, 2), textcoords="offset points")
    ax.set_xlabel("skin: log2FC malignant vs reactive")
    ax.set_ylabel("blood: log2FC malignant vs reactive")
    ax.set_title(f"{len(up_both)} antigens selective in skin and blood",
                 loc="left", fontsize=7.5)
    TF._clean(ax)
    fig.tight_layout()
    TF.save(fig, "skin_vs_blood", caption_text=(
        "Red: validated antigens and disputed genes. Both compartments artifact-tested."))
    up_both.to_csv(ST.tab("shared_skin_blood"), index=False)
    TF.logfig("skin_vs_blood", "skin vs blood selectivity", len(both))
    print("\nwrote", ST.tab("shared_skin_blood"))

In [ ]:
# ============================================================================
# §9  Dossier
# ============================================================================
def SEC(t): print("\n" + "=" * 78 + f"\n{t}\n" + "=" * 78)
def kv(k, v): print(f"  {k:34s} {v}")

SEC("1 - INPUT")
kv("object", Path(man["source"]).name)
kv("builder", man.get("builder"))
kv("genes in object", f"{man['n_genes_in_object']:,}"
   + ("  *** HVG subset ***" if man["n_genes_in_object"] < 20000 else ""))
kv("malignancy call", man["malignancy_call"])
kv("library for CP10K", man.get("library"))

SEC("2 - COHORT")
kv("donor keys in the object", len(don))
kv("dropped: ex vivo drug arms", int(don["drop_reason"].eq("ex vivo drug-culture arm").sum()))
kv("dropped: duplicate persons",
   int(don["drop_reason"].fillna("").str.startswith("same person").sum()))
kv("independent keys", int(don["cohort_keep"].sum()))
kv("of those, 2-day cultured", int((don["cohort_keep"] & don["is_cultured"]).sum()))
kv("healthy donors moved to CD4_HC", int((don["is_hc"] & don["cohort_keep"]).sum()))
kv("donors with both CD4 arms", len(PAIRED_DONORS))
print()
print(pd.crosstab(don.loc[don["cohort_keep"], "dx"],
                  don.loc[don["cohort_keep"], "study"]).to_string())

SEC("3 - ARMS"); print(arms.to_string())

SEC("4 - ARTIFACT TESTS")
kv("lineage controls found", band.attrs["n_genes"])
kv("null calibrated", ok)
print("  " + msg)
print()
print(ev["artifact_flag"].value_counts().to_string())
kv("\n  genes excluded from the headline",
   int((ev["artifact_flag"].ne("clean") & (ev["log2fc_paired"] > 0)).sum()))

SEC("5 - CALIBRATION"); print(hit.to_string(index=False))
if missing:
    print(f"  not testable (HVG): {missing}")

SEC("6 - WHAT THIS IS NOT")
for line in [
    "A full-gene screen: the object is the 10k HVG subset, and CD4 and TNFRSF8 are not in it.",
    "A myeloid or HSPC safety statement: those compartments are not in `long` at all.",
    "A statement about the CD8 or NK arm: nb31 labelled on CD8A and that arm is a mixture.",
    "A target call. Nothing here has been seen as protein; 45 does that for 17 Sezary donors.",
    "A full-gene comparison with skin: the skin object carries all 42,347 genes and this "
    "one does not, so §8 compares only what both objects hold.",
]:
    print("  - " + line)

SEC("7 - OUTPUTS")
TF.write_index("figure_index")
for p in sorted(ST.TAB_DIR.glob(f"{PREFIX}*")) + sorted(ST.FIG_DIR.glob(f"{PREFIX}*")):
    print("  ", p.relative_to(NB_DIR))

In [ ]:
# ============================================================================
# §10  The complete numerical result, as one block to hand to another model
# ============================================================================
_ctx = f"""
Compartment: MF/CTCL **blood**, from {Path(man['source']).name}
({man['n_genes_in_object']:,} genes in the object -- an HVG SUBSET, so a gene absent from the
screen is not necessarily unexpressed; CD4 and TNFRSF8/CD30 are both missing for this reason).

Cohort: {len(don)} donor keys reduced to {int(don['cohort_keep'].sum())} independent keys by
dropping the geskin2026 ex vivo drug-culture wells and duplicate persons, and by moving five
healthy donors that the atlas labelled SS out of the reactive arm. Diagnosis from the curated
`clin_dx`. {len(PAIRED_DONORS)} donors carry both a malignant and a reactive CD4 arm.

Arms: malignancy call `{man['malignancy_call']}`. Blood in Sezary syndrome is ~92% malignant
CD4, so the reactive arm is small in many donors. The CD8 arm is a CD8+NK mixture (nb31 labelled
on CD8A) and any statement about it is a statement about a mixture.

Headline test: per gene, the within-donor difference of mean log1p CP10K between the malignant
and reactive arms, paired Wilcoxon over donors, BH-adjusted (`q`). `log2fc_paired` is the median
per-donor log2 ratio; `frac_donors_up` the share of donors with a positive difference.

Artifact tests: six, against an empirical null made of transcripts a CD4 T cell cannot make
(LYZ, S100A8/9, CD14, VCAN, FCN1, CST3, LST1, AIF1, TYROBP, FCER1G, CD68, MNDA, PPBP, PF4, HBB,
MS4A1, CD79A). Thresholds are read off that null rather than chosen, and the section is allowed
to exclude nothing: the calibration verdict decides. Verdict here: {msg}

Known answers, for calibration: CCR4 (mogamulizumab) is approved in CTCL, KIR3DL2 is lacutamab's
target, IL2RA is registrational; TNFRSF8/CD30 is absent from this object. CD7 and DPP4/CD26 loss
is the Sezary immunophenotype and must come out negative.

Scope limits that apply to every number: the off-tumour axis covers T and NK arms ONLY -- the
non-T blood compartments are not in the screen table, so a myeloid or HSPC hit, the kind that
causes cytopenia, is invisible here. Nothing below has been seen as protein; notebook 45 does
that for 17 Sezary donors with CITE-seq.
"""

_sec = [
    {"name": "§0b Cohort", "what": "the independent donor set and its clinical fields.",
     "how": "curated clin_* columns only; no expression used.",
     "table": don[don["cohort_keep"]][[c for c in ("study", "dx", "dx2", "is_cultured",
                                                   "clone_shared", "sex", "age", "n_cells",
                                                   "burden") if c in don]],
     "file": "tables/tgt43_blood_cohort.csv", "max_rows": 100},
    {"name": "§0b Dropped keys", "what": "the keys removed, with the reason.",
     "how": "ex vivo drug-culture arm, or the same person under another key.",
     "table": don.loc[~don["cohort_keep"], ["study", "dx", "clin_ex_vivo", "person", "n_cells",
                                            "drop_reason"]], "max_rows": 60},
    {"name": "§1 Arms", "what": "donors, cells and studies per arm after the cohort fix.",
     "how": "groups of >= 20 cells.", "table": arms},
    {"name": "§1 Contamination load per arm",
     "what": "how dirty each arm is -- the asymmetry a contamination artifact would need.",
     "how": "median over the arm's donors.",
     "table": meta.groupby("arm")[["frac_lineage_pos", "mean_myeloid_score",
                                   "median_total_counts", "median_doublet_score"]].median()},
    {"name": "§2a Contamination null",
     "what": "what pure contamination scores in this cohort, per statistic.",
     "how": "the same tests applied to the lineage-control genes; thresholds come from the p90 "
            "row.", "table": band},
    {"name": "§2a Calibration verdict",
     "what": "whether the differential-contamination mechanism operates here at all.",
     "how": "the controls must themselves be up in the malignant arm; if they are not, no gene "
            "is excluded.", "table": {"null_calibrated": ok, "verdict": msg}},
    {"name": "§2b Artifact verdict, per gene",
     "what": "the six tests for every gene up at q < 0.05.",
     "how": "retained_* = share of the effect surviving trimming / winsorising / dropping dirty "
            "cells / depth matching; *_excess = prevalence-matched residual against the null; "
            "rho_burden = Spearman of the per-donor effect against that donor's "
            "malignant:reactive ratio. Two failures = artifact, one = suspect.",
     "table": ev[(ev["q"] < 0.05) & (ev["log2fc_paired"] > 0)].sort_values(
         "log2fc_paired", ascending=False)[[c for c in _fc if c in ev]],
     "file": "tables/tgt43_artifact_evidence.csv", "max_rows": 80},
    {"name": "§2c Three-arm detection",
     "what": "median detection of the disputed genes and the controls in healthy, reactive and "
             "malignant CD4.",
     "how": "median over donors of the share of cells with >= 1 UMI. Contamination has no "
            "reason to spare healthy CD4.", "table": det},
    {"name": "§3 Candidates passing every gate",
     "what": "the genes selectively up on malignant CD4 that survived §2.",
     "how": "clean, q < 0.05, paired log2FC > 0.5, detected in >= 50% of donors, not a loss "
            "marker or lineage control.",
     "table": up[[c for c in _c if c in up]]},
    {"name": "§3 Loss markers",
     "what": "the Sezary loss phenotype, as the check that cleaning did not flatten real "
             "biology.", "how": "same paired test.",
     "table": core[core["gene"].isin(ST.LOSS_MARKERS)][["gene", "log2fc_paired",
                                                        "frac_donors_up", "q",
                                                        "artifact_flag"]]},
    {"name": "§3 FD Per-gene summary of the donor x gene matrix",
     "what": "how consistent each leading antigen is across patients.",
     "how": "per-donor log2FC; frac_up_1 is the share of donors where the antigen is at least "
            "doubled.", "table": _sum},
    {"name": "§3 FE Eligibility",
     "what": "the fraction of patients in whom at least X% of malignant CD4 cells are positive.",
     "how": "transcript detection per donor, thresholded.", "table": elig,
     "file": "tables/tgt43_eligibility_blood.csv"},
    {"name": "§4 Off-tumour, T and NK arms only",
     "what": "the worst non-target T arm and the healthy-CD4 burden for each candidate.",
     "how": "share of cells detected. Monocytes, DC, B, plasma, platelets and HSPC are NOT in "
            "this table.", "table": _o},
    {"name": "§5 Trafficking modules per arm",
     "what": "the state of the machinery that puts a protein on the surface.",
     "how": "per-module z across donors, then the arm mean. Context, not a correction.",
     "table": arm_mod},
    {"name": "§6 Escape risk",
     "what": "how much each candidate varies between patients and within one patient's clusters.",
     "how": "SD of detection across donors, and across a donor's own clusters.",
     "table": esc[esc["gene"].isin(SHORT_GENES)]},
    {"name": "§7 Control check",
     "what": "where the antigens with a known answer sit in the ranking.",
     "how": "rank by paired log2FC over all ranked genes. TNFRSF8 is not testable: it is absent "
            "from the HVG object.", "table": hit},
    {"name": "§7 Shortlist with the plausibility layer",
     "what": "the final candidate table, with surfaceome evidence beside the statistics.",
     "how": "surface_confident / n_tm / surfy_class come from CSPA + Surfy, not from this "
            "cohort.", "table": short[[c for c in show if c in short]],
     "file": "tables/tgt43_shortlist_blood.csv"},
]
if "both_calls" in dir():
    _sec.append({"name": "§2d TCR-call control",
                 "what": "the same screen with the arms built from the expanded TCR clone "
                         "instead of the Leiden cluster.",
                 "how": "an effect that is an artifact of how the clusters were drawn cannot "
                        "survive a call made from the TCR.",
                 "table": both_calls[both_calls["gene"].isin(
                     TQ.SUSPECTS + ST.POSITIVE_CONTROLS + ST.LOSS_MARKERS)],
                 "file": "tables/tgt43_call_sensitivity.csv"})
if "up_both" in dir():
    _sec.append({"name": "§8 Selective in skin AND blood",
                 "what": "antigens that separate tumour from reactive CD4 in both compartments.",
                 "how": "q < 0.05 and log2FC > 0.5 on both sides, after both screens were "
                        "cohort-fixed and artifact-tested.",
                 "table": up_both[[c for c in cols if c in up_both]],
                 "file": "tables/tgt43_shared_skin_blood.csv"})
TF.llm_export("43", "Surface-target screen, blood", _ctx, _sec)